# 🧠 Advanced Prompt Engineering Techniques — Live Demo

### Using OpenAI GPT API (gpt-4o / gpt-3.5-turbo)

This notebook demonstrates **11 advanced prompting techniques** with real API calls, detailed explanations, visual comparisons, and production-ready code.

| # | Technique | Category | Key Insight |
|---|-----------|----------|-------------|
| 1 | Zero-Shot | Fundamental | No examples needed |
| 2 | Few-Shot | Fundamental | Learn from demonstrations |
| 3 | Chain-of-Thought (CoT) | Reasoning | Step-by-step with examples |
| 4 | Zero-Shot CoT | Reasoning | "Let's think step by step" |
| 5 | Self-Consistency | Advanced | Multiple paths + majority vote |
| 6 | Tree of Thoughts (ToT) | Advanced | Branch, evaluate, backtrack |
| 7 | ReAct | Agentic | Reasoning + Action loops |
| 8 | Role / Persona | Persona | Expert identity steering |
| 9 | Meta Prompting | Meta | LLM writes its own prompts |
| 10 | Structured Output (JSON) | Production | Guaranteed parseable output |
| 11 | Prompt Chaining | Production | Multi-step pipelines |

---

## ⚙️ Setup & Configuration

In [1]:
# ═══════════════════════════════════════════════════════
#  STEP 1: Install dependencies
# ═══════════════════════════════════════════════════════
%pip install openai -q
print("✅ OpenAI package installed.")

✅ OpenAI package installed.


In [ ]:
# ═══════════════════════════════════════════════════════
#  STEP 2: Enter your API key
# ═══════════════════════════════════════════════════════
#
#  Option A: Use Google Colab's secrets (recommended)
#    1. Click the 🔑 key icon in the left sidebar
#    2. Add a secret named OPENAI_API_KEY
#    3. Paste your key as the value
#
#  Option B: Paste directly below (less secure)
# ═══════════════════════════════════════════════════════

import os

# Try Colab secrets first
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

# If still empty, prompt the user
if not OPENAI_API_KEY:
    import getpass
    OPENAI_API_KEY = getpass.getpass("🔑 Enter your OpenAI API key: ")

assert OPENAI_API_KEY, "❌ No API key provided!"
print(f"✅ API key set (ending in ...{OPENAI_API_KEY[-4:]})")

🔑 Enter your OpenAI API key: ··········
✅ API key set (ending in ...lFMA)


In [16]:
# ═══════════════════════════════════════════════════════
#  STEP 3: Configuration & Helper Functions
# ═══════════════════════════════════════════════════════

import json
import time
import textwrap
from collections import Counter
from openai import OpenAI
from IPython.display import display, HTML, Markdown, clear_output

# ── Config ──────────────────────────────────────────
MODEL = "gpt-4o"               # Change to "gpt-3.5-turbo" for cheaper runs
# MODEL = "gpt-4o-mini"        # Good balance of cost and quality
# MODEL = "gpt-3.5-turbo"      # Cheapest option
MAX_TOKENS = 4024
DEFAULT_TEMP = 0.7

# ── Client ─────────────────────────────────────────
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Token / Cost tracker ──────────────────────────
usage_log = {"calls": 0, "tokens": 0, "cost": 0.0}

PRICING = {
    "gpt-4o":        {"input": 2.50 / 1e6, "output": 10.00 / 1e6},
    "gpt-4o-mini":   {"input": 0.15 / 1e6, "output": 0.60 / 1e6},
    "gpt-3.5-turbo": {"input": 0.50 / 1e6, "output": 1.50 / 1e6},
}


def call_llm(messages, model=None, temperature=None, max_tokens=None,
             n=1, response_format=None):
    """
    Call OpenAI API with tracking.
    Returns dict with: content, model, usage, elapsed, finish_reason
    """
    model = model or MODEL
    temperature = temperature if temperature is not None else DEFAULT_TEMP
    max_tokens = max_tokens or MAX_TOKENS

    kwargs = dict(model=model, messages=messages, temperature=temperature,
                  max_tokens=max_tokens, n=n)
    if response_format:
        kwargs["response_format"] = response_format

    start = time.time()
    resp = client.chat.completions.create(**kwargs)
    elapsed = round(time.time() - start, 2)

    content = (resp.choices[0].message.content if n == 1
               else [c.message.content for c in resp.choices])

    u = resp.usage
    usage = {"prompt": u.prompt_tokens, "completion": u.completion_tokens,
             "total": u.total_tokens}

    p = PRICING.get(model, PRICING["gpt-4o"])
    cost = u.prompt_tokens * p["input"] + u.completion_tokens * p["output"]

    usage_log["calls"] += 1
    usage_log["tokens"] += usage["total"]
    usage_log["cost"] += cost

    return {"content": content, "model": resp.model, "usage": usage,
            "elapsed": elapsed, "cost": cost,
            "finish_reason": resp.choices[0].finish_reason}


# ── Display Helpers ────────────────────────────────

def show_banner(num, title, tag, color, description):
    display(HTML(f"""
    <div style="margin:30px 0 10px; padding:20px 24px; background:#0e1117;
         border-left:4px solid {color}; border-radius:0 12px 12px 0;
         font-family:system-ui,sans-serif;">
      <span style="font-size:11px; font-weight:700; letter-spacing:2px;
           color:{color}; text-transform:uppercase;">{tag}</span>
      <h2 style="margin:6px 0 8px; color:#fff; font-size:22px;">
        {num}. {title}</h2>
      <p style="color:#8b95a5; margin:0; font-size:14px; line-height:1.6;">
        {description}</p>
    </div>
    """))


def show_prompt(label, content, color="#fbbf24"):
    escaped = content.strip().replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    display(HTML(f"""
    <div style="margin:12px 0; font-family:system-ui,sans-serif;">
      <div style="font-size:12px; font-weight:700; color:{color};
           letter-spacing:1px; margin-bottom:6px;">📋 {label}</div>
      <div style="background:#161b22; border:1px solid #30363d;
           border-radius:8px; padding:14px 16px; overflow-x:auto;">
        <pre style="margin:0; font-size:12.5px; line-height:1.7;
             color:#c9d1d9; white-space:pre-wrap; word-break:break-word;
             font-family:'Courier New',monospace;">{escaped}</pre>
      </div>
    </div>
    """))


def show_response(content, result=None, label="Model Response", color="#4ade80"):
    escaped = content.strip().replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    meta = ""
    if result:
        meta = (f'<span style="color:#4a5568; font-size:11px; float:right;">'
                f'{result["elapsed"]}s · {result["usage"]["total"]} tokens · '
                f'${result["cost"]:.4f}</span>')
    display(HTML(f"""
    <div style="margin:12px 0; font-family:system-ui,sans-serif;">
      <div style="font-size:12px; font-weight:700; color:{color};
           letter-spacing:1px; margin-bottom:6px;">✅ {label} {meta}</div>
      <div style="background:#0d1117; border:1px solid #1a3a2a;
           border-radius:8px; padding:14px 16px; overflow-x:auto;">
        <pre style="margin:0; font-size:12.5px; line-height:1.7;
             color:#b0d4b8; white-space:pre-wrap; word-break:break-word;
             font-family:'Courier New',monospace;">{escaped}</pre>
      </div>
    </div>
    """))


def show_comparison(items, title="Comparison"):
    """items = [(label, content, color), ...]"""
    cols = ""
    w = max(30, 95 // len(items))
    for label, content, color in items:
        escaped = content.strip()[:600].replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        cols += f"""
        <div style="flex:1; min-width:280px; background:#0d1117;
             border:1px solid #30363d; border-radius:8px; overflow:hidden;">
          <div style="padding:8px 12px; background:#161b22;
               border-bottom:1px solid #30363d; font-size:12px;
               font-weight:700; color:{color};">{label}</div>
          <pre style="margin:0; padding:12px; font-size:11.5px; line-height:1.6;
               color:#c9d1d9; white-space:pre-wrap; word-break:break-word;
               font-family:'Courier New',monospace; max-height:350px;
               overflow-y:auto;">{escaped}</pre>
        </div>"""
    display(HTML(f"""
    <div style="margin:16px 0; font-family:system-ui,sans-serif;">
      <div style="font-size:13px; font-weight:700; color:#a78bfa;
           margin-bottom:8px;">⚖️ {title}</div>
      <div style="display:flex; gap:10px; flex-wrap:wrap;">{cols}</div>
    </div>
    """))


def show_tip(text):
    display(HTML(f"""
    <div style="margin:14px 0; padding:12px 16px; background:rgba(74,222,128,0.06);
         border-left:3px solid #4ade80; border-radius:0 8px 8px 0;
         font-size:13px; color:#b0d4b8; line-height:1.6;
         font-family:system-ui,sans-serif;">
      💡 <strong style="color:#4ade80;">Tip:</strong> {text}
    </div>
    """))


def show_warning(text):
    display(HTML(f"""
    <div style="margin:14px 0; padding:12px 16px; background:rgba(251,146,60,0.06);
         border-left:3px solid #fb923c; border-radius:0 8px 8px 0;
         font-size:13px; color:#f0c8a0; line-height:1.6;
         font-family:system-ui,sans-serif;">
      ⚠️ <strong style="color:#fb923c;">Note:</strong> {text}
    </div>
    """))


def show_diagram(html_content):
    display(HTML(f"""
    <div style="margin:16px 0; padding:20px; background:#0e1117;
         border:1px solid #30363d; border-radius:10px;
         font-family:'Courier New',monospace; font-size:13px;
         text-align:center; color:#c9d1d9;">{html_content}</div>
    """))


# ── Quick connectivity test ────────────────────────
try:
    test = client.models.list()
    print(f"✅ Connected to OpenAI. Using model: {MODEL}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to OpenAI. Using model: gpt-4o


---
## 🚀 Run All Demos

Each section below is a **self-contained demo**. Run them top-to-bottom, or jump to any technique.

---
## 1. ⚡ Zero-Shot Prompting

In [4]:
show_banner(1, "Zero-Shot Prompting", "FUNDAMENTAL", "#4ade80",
    "No examples provided. The model relies entirely on its pre-trained "
    "knowledge to understand and complete the task from the instruction alone.")

# ── Demo 1A: Classification ─────────────────────────
prompt_1a = """Classify each sentence as FACT, OPINION, or QUESTION.

1. The Earth orbits the Sun.
2. Pizza is the best food in the world.
3. How many planets are in the solar system?
4. Shakespeare wrote Hamlet.
5. The movie was incredibly boring."""

show_prompt("Zero-Shot Classification", prompt_1a)

r1a = call_llm([
    {"role": "system", "content": "You are a precise text classifier. Respond with only the number and label."},
    {"role": "user", "content": prompt_1a}
], temperature=0)
show_response(r1a["content"], r1a)

# ── Demo 1B: Structured Extraction ──────────────────
prompt_1b = """Extract all person names, locations, and dates from this text.
Return as JSON with keys: "persons", "locations", "dates".

"Dr. Sarah Chen presented her findings at the Tokyo Conference on March 15, 2024.
Her colleague James Miller, based in London, co-authored the paper.
They plan to visit MIT in Cambridge next September."""

show_prompt("Zero-Shot Entity Extraction → JSON", prompt_1b)

r1b = call_llm([
    {"role": "system", "content": "You are a named entity extractor. Return only valid JSON."},
    {"role": "user", "content": prompt_1b}
], temperature=0)
show_response(r1b["content"], r1b)

show_tip("Zero-shot works best for well-defined tasks (classification, extraction, translation). "
         "For nuanced or ambiguous tasks, add examples (few-shot).")

---
## 2. 📝 Few-Shot Prompting

In [5]:
show_banner(2, "Few-Shot Prompting", "FUNDAMENTAL", "#60a5fa",
    "Provide 2-5 input→output examples to teach the model the desired "
    "pattern, format, and reasoning style through demonstration.")

# ── Demo 2A: Custom sentiment scale ─────────────────
prompt_2a = """Rate product reviews on this scale: TERRIBLE / BAD / NEUTRAL / GOOD / AMAZING

Review: "Absolutely love this! Best purchase I've made all year."
Rating: AMAZING

Review: "It works fine, nothing special. Does what it says."
Rating: NEUTRAL

Review: "Broke after two days. Complete waste of money."
Rating: TERRIBLE

Review: "Pretty good quality, though the color was slightly off."
Rating: GOOD

Now rate these:

Review: "The delivery was fast but the product smells like chemicals and gave me a headache."
Rating:

Review: "Exceeded all my expectations. The build quality is premium and customer support was incredible."
Rating:

Review: "It's okay I guess. Not great, not terrible. Probably won't buy again."
Rating:"""

show_prompt("Few-Shot — Custom Sentiment Scale (4 examples)", prompt_2a)

r2a = call_llm([{"role": "user", "content": prompt_2a}], temperature=0)
show_response(r2a["content"], r2a)

# ── Demo 2B: Style transfer ─────────────────────────
prompt_2b = """Rewrite technical jargon as simple analogies a 10-year-old would understand.

Technical: "The CPU executes instructions from RAM using a fetch-decode-execute cycle."
Simple: "The brain of the computer reads recipes from a notepad — it reads one step, figures out what it means, does it, then moves to the next step, over and over really fast."

Technical: "DNS resolves domain names to IP addresses using a distributed hierarchical database."
Simple: "DNS is like a giant phone book for the internet. When you type 'google.com', it looks up the real address number so your computer knows where to go."

Technical: "Machine learning models minimize a loss function through gradient descent by iteratively adjusting weights in the direction of steepest descent."
Simple:"""

show_prompt("Few-Shot — Jargon → Kid-Friendly Analogy", prompt_2b)

r2b = call_llm([{"role": "user", "content": prompt_2b}], temperature=0.7)
show_response(r2b["content"], r2b)

show_tip("Quality > Quantity: 3-5 diverse, well-crafted examples beat 10+ mediocre ones. "
         "The model learns format, tone, AND reasoning from your demonstrations.")

---
## 3. 🔗 Chain-of-Thought (CoT)

In [17]:
show_banner(3, "Chain-of-Thought (CoT)", "REASONING", "#a78bfa",
    "Show step-by-step reasoning in examples so the model learns to 'think aloud' "
    "before answering. Dramatically improves accuracy on math, logic, and multi-step problems.")

problem = ("A store has a 'Buy 2 Get 1 Free' deal on shirts that cost $25 each. "
           "Sales tax is 8%. If Maria buys 7 shirts, how much does she pay in total?")

# ── WITHOUT CoT ─────────────────────────────────────
show_prompt("WITHOUT CoT (Direct Answer)", problem)
r3_no_cot = call_llm([
    {"role": "system", "content": "Answer the math question. Be concise."},
    {"role": "user", "content": problem}
], temperature=0)

# ── WITH CoT ────────────────────────────────────────
cot_prompt = f"""Solve math problems step by step. Here's an example:

Q: A store offers "Buy 3 Get 1 Free" on $10 items. Tax is 10%. If John buys 5 items, how much?
Step 1: For every 3 items bought, 1 is free. 5 items → 1 group of (3 paid + 1 free) = 4 items, + 1 more paid = 5 items. Free items = 1.
Step 2: Paid items = 5 - 1 = 4 items.
Step 3: Subtotal = 4 × $10 = $40
Step 4: Tax = $40 × 0.10 = $4
Step 5: Total = $40 + $4 = $44
Answer: $44

Now solve this step by step:
Q: {problem}"""

show_prompt("WITH CoT (Step-by-Step Example)", cot_prompt)
r3_cot = call_llm([{"role": "user", "content": cot_prompt}], temperature=0)

# ── Side by side ────────────────────────────────────
show_comparison([
    ("❌ Without CoT", r3_no_cot["content"], "#f87171"),
    ("✅ With CoT", r3_cot["content"], "#4ade80"),
], "Direct Answer vs Chain-of-Thought")

show_tip("Correct answer: 7 shirts → 2 free (two groups of Buy2Get1), 5 paid. "
         "5 × $25 = $125, tax = $10, total = <b>$135</b>. "
         "CoT makes errors auditable — you can check each step.")

---
## 4. 💭 Zero-Shot Chain-of-Thought

In [18]:
show_banner(4, "Zero-Shot Chain-of-Thought", "REASONING", "#22d3ee",
    'Just append "Let\'s think step by step" — no examples needed. '
    'The model auto-decomposes the problem. Discovered by Kojima et al. (2022).')

problem = ("A farmer has 15 chickens. All but 8 die. Then he buys twice as many "
           "chickens as he currently has. A fox takes 3 chickens. Each remaining "
           "chicken lays 4 eggs. Half the eggs hatch. How many baby chicks are there?")

# ── Direct ──────────────────────────────────────────
show_prompt("Direct (no trigger phrase)", problem)
r4_direct = call_llm([{"role": "user", "content": problem}], temperature=0)

# ── With magic phrase ───────────────────────────────
cot_problem = problem + "\n\nLet's think step by step."
show_prompt('With "Let\'s think step by step"', cot_problem)
r4_cot = call_llm([{"role": "user", "content": cot_problem}], temperature=0)

show_comparison([
    ("Direct Answer", r4_direct["content"], "#f87171"),
    ('"Let\'s think step by step"', r4_cot["content"], "#22d3ee"),
], "The Magic Phrase Effect")

# ── Other trigger phrases ───────────────────────────
display(HTML("""
<div style="margin:16px 0; font-family:system-ui,sans-serif;">
  <div style="font-size:13px; font-weight:700; color:#22d3ee; margin-bottom:10px;">
    🔮 Other Effective CoT Trigger Phrases</div>
  <table style="width:100%; border-collapse:collapse; font-size:13px; color:#c9d1d9;">
    <tr style="background:#161b22;">
      <th style="text-align:left; padding:8px 12px; color:#22d3ee;">Phrase</th>
      <th style="text-align:left; padding:8px 12px; color:#22d3ee;">Best For</th></tr>
    <tr><td style="padding:8px 12px; border-top:1px solid #30363d;">"Let's think step by step."</td>
        <td style="padding:8px 12px; border-top:1px solid #30363d;">General reasoning</td></tr>
    <tr><td style="padding:8px 12px; border-top:1px solid #30363d;">"Let's work through this carefully."</td>
        <td style="padding:8px 12px; border-top:1px solid #30363d;">Complex multi-step</td></tr>
    <tr><td style="padding:8px 12px; border-top:1px solid #30363d;">"Break this down into steps."</td>
        <td style="padding:8px 12px; border-top:1px solid #30363d;">Decomposition tasks</td></tr>
    <tr><td style="padding:8px 12px; border-top:1px solid #30363d;">"Show your reasoning."</td>
        <td style="padding:8px 12px; border-top:1px solid #30363d;">Math / logic</td></tr>
    <tr><td style="padding:8px 12px; border-top:1px solid #30363d;">"First, let's identify what we know."</td>
        <td style="padding:8px 12px; border-top:1px solid #30363d;">Word problems</td></tr>
  </table>
</div>
"""))

show_tip("Correct: 8 survive → buy 16 more → 24 → fox takes 3 → 21 → 84 eggs → 42 hatch = <b>42 baby chicks</b>.")

Phrase,Best For
"""Let's think step by step.""",General reasoning
"""Let's work through this carefully.""",Complex multi-step
"""Break this down into steps.""",Decomposition tasks
"""Show your reasoning.""",Math / logic
"""First, let's identify what we know.""",Word problems


---
## 5. 🎯 Self-Consistency

In [8]:
show_banner(5, "Self-Consistency", "ADVANCED", "#fb923c",
    "Sample MULTIPLE reasoning paths (with temperature > 0) and take a "
    "MAJORITY VOTE on the final answer. Different chains make different "
    "mistakes, but the correct answer tends to be the most common.")

# ── Diagram ─────────────────────────────────────────
show_diagram("""
  <div style="font-weight:700; font-size:14px; color:#fff; margin-bottom:16px;">
    Self-Consistency: Sample → Vote → Answer</div>
  <div style="color:#fb923c; font-size:15px;">Same Prompt (temp=0.9)</div>
  <div style="font-size:20px; margin:8px 0;">↙ &nbsp; ↓ &nbsp; ↓ &nbsp; ↓ &nbsp; ↘</div>
  <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
    <span style="background:rgba(167,139,250,0.15); padding:6px 10px; border-radius:6px; color:#a78bfa;">Path 1</span>
    <span style="background:rgba(96,165,250,0.15); padding:6px 10px; border-radius:6px; color:#60a5fa;">Path 2</span>
    <span style="background:rgba(244,114,182,0.15); padding:6px 10px; border-radius:6px; color:#f472b6;">Path 3</span>
    <span style="background:rgba(74,222,128,0.15); padding:6px 10px; border-radius:6px; color:#4ade80;">Path 4</span>
    <span style="background:rgba(251,191,36,0.15); padding:6px 10px; border-radius:6px; color:#fbbf24;">Path 5</span>
  </div>
  <div style="font-size:20px; margin:8px 0;">↘ &nbsp; ↘ &nbsp; ↓ &nbsp; ↙ &nbsp; ↙</div>
  <div style="display:inline-block; background:rgba(74,222,128,0.15); padding:8px 20px;
       border-radius:8px; color:#4ade80; font-weight:700;">✓ Majority Vote → Final Answer</div>
""")

problem = "When I was 6, my sister was half my age. Now I'm 70. How old is my sister?"
cot_prompt = f"{problem}\n\nLet's think step by step."

show_prompt("Problem (sampled 5× with temperature=0.9)", cot_prompt)

# ── Sample 5 paths ─────────────────────────────────
print("⏳ Sampling 5 reasoning paths...")
paths = []
for i in range(5):
    r = call_llm([{"role": "user", "content": cot_prompt}], temperature=0.9)
    paths.append(r["content"])
    print(f"   Path {i+1} ✓")
    time.sleep(0.5)

# ── Display all paths ──────────────────────────────
colors = ["#a78bfa", "#60a5fa", "#f472b6", "#4ade80", "#fbbf24"]
for i, (path, color) in enumerate(zip(paths, colors)):
    show_response(path, label=f"Path {i+1}", color=color)

# ── Extract & vote ─────────────────────────────────
# Ask the model to extract just the number from each path
import re
extracted_answers = []
for path in paths:
    nums = re.findall(r'\b(\d{1,2})\b', path.split('\n')[-1] if path.strip() else path)
    # Try to find the final answer number
    answer_line = [l for l in path.strip().split('\n') if any(w in l.lower() for w in ['answer', 'sister is', 'old'])]
    if answer_line:
        nums = re.findall(r'\b(\d{1,3})\b', answer_line[-1])
    extracted_answers.append(nums[-1] if nums else "?")

vote_counts = Counter(extracted_answers)
winner = vote_counts.most_common(1)[0]

display(HTML(f"""
<div style="margin:16px 0; padding:16px 20px; background:#0e1117;
     border:2px solid #fb923c; border-radius:10px; font-family:system-ui,sans-serif;">
  <div style="font-size:14px; font-weight:700; color:#fb923c; margin-bottom:10px;">🗳️ Majority Vote Results</div>
  <div style="color:#c9d1d9; font-size:13px; margin-bottom:8px;">
    Extracted answers: {', '.join(f'Path {i+1}→<b>{a}</b>' for i, a in enumerate(extracted_answers))}</div>
  <div style="color:#c9d1d9; font-size:13px; margin-bottom:8px;">
    Vote counts: {dict(vote_counts)}</div>
  <div style="color:#4ade80; font-size:16px; font-weight:700;">✅ Winner: {winner[0]} ({winner[1]}/5 votes)</div>
  <div style="color:#8b95a5; font-size:12px; margin-top:6px;">
    Correct answer: 67 (when I was 6, sister was 3 → always 3 years younger → 70-3 = 67)</div>
</div>
"""))

show_warning("Self-Consistency costs N× more tokens (here 5×). Use it for high-stakes "
             "decisions where correctness matters more than cost.")

⏳ Sampling 5 reasoning paths...
   Path 1 ✓
   Path 2 ✓
   Path 3 ✓
   Path 4 ✓
   Path 5 ✓


---
## 6. 🌳 Tree of Thoughts (ToT)

In [9]:
show_banner(6, "Tree of Thoughts (ToT)", "ADVANCED", "#f472b6",
    "Explore multiple reasoning BRANCHES at each step, EVALUATE them, "
    "PRUNE dead ends, and BACKTRACK. Mimics how humans solve puzzles — "
    "trying options, hitting dead ends, and reconsidering.")

# ── Diagram ─────────────────────────────────────────
show_diagram("""
  <div style="font-weight:700; font-size:14px; color:#fff; margin-bottom:12px;">Tree of Thoughts: Explore → Evaluate → Expand Best</div>
  <div style="color:#f472b6;">🧩 Problem</div>
  <div style="font-size:18px; margin:6px 0;">↙ &nbsp;&nbsp; ↓ &nbsp;&nbsp; ↘</div>
  <div style="display:flex; gap:8px; justify-content:center;">
    <span style="background:rgba(74,222,128,0.15); padding:6px 10px; border-radius:6px; color:#4ade80;">Branch A ✓</span>
    <span style="background:rgba(96,165,250,0.15); padding:6px 10px; border-radius:6px; color:#60a5fa;">Branch B ✓</span>
    <span style="background:rgba(248,113,113,0.15); padding:6px 10px; border-radius:6px; color:#f87171;">Branch C ✗</span>
  </div>
  <div style="font-size:14px; margin:6px 0; color:#8b95a5;">evaluate & prune ↓</div>
  <div style="display:flex; gap:8px; justify-content:center;">
    <span style="background:rgba(74,222,128,0.15); padding:6px 10px; border-radius:6px; color:#4ade80;">A → A1 ✓✓</span>
    <span style="background:rgba(248,113,113,0.1); padding:6px 10px; border-radius:6px; color:#f87171; text-decoration:line-through;">A → A2 ✗</span>
    <span style="background:rgba(96,165,250,0.1); padding:6px 10px; border-radius:6px; color:#60a5fa;">B → B1 ✓</span>
  </div>
  <div style="font-size:18px; margin:8px 0;">↓</div>
  <div style="display:inline-block; background:rgba(74,222,128,0.2); padding:8px 20px;
       border-radius:8px; color:#4ade80; font-weight:700;">★ Best Path: A → A1 → Solution</div>
""")

tot_prompt = """Solve this creative puzzle using Tree of Thoughts exploration.

PUZZLE: Using the numbers 3, 4, 7, 8 and operations +, -, ×, ÷,
make the number 24. Each number must be used exactly once.

INSTRUCTIONS — Explore like a tree:

LEVEL 1: Generate 3 possible first operations (pick any two numbers to combine).
For each, evaluate: does this look promising? Rate as [PROMISING] or [DEAD END].

LEVEL 2: For each PROMISING branch, generate 2 possible next operations.
Evaluate each again.

LEVEL 3: Continue the most promising branches to completion.

BACKTRACK: If a branch can't reach 24, explicitly mark it ✗ and try another.

Show your full exploration tree with clear formatting, then state the final solution."""

show_prompt("Tree of Thoughts — Game of 24", tot_prompt)

r6 = call_llm([
    {"role": "system", "content": "You are a methodical problem solver. Show your tree exploration "
     "with branches, evaluations, pruning, and backtracking. Use clear formatting."},
    {"role": "user", "content": tot_prompt}
], temperature=0.3, max_tokens=1500)

show_response(r6["content"], r6)

show_tip("ToT excels at puzzles, planning, creative writing, and strategic tasks. "
         "In production, implement as multiple API calls: generate → evaluate → expand best → repeat.")

---
## 7. 🔄 ReAct Prompting (Reasoning + Action)

In [10]:
show_banner(7, "ReAct Prompting", "AGENTIC", "#fbbf24",
    "Reasoning + Action loop. The model THINKS about what to do, "
    "ACTS (searches, calculates), OBSERVES results, and repeats. "
    "This is the foundation of modern AI agent frameworks.")

react_prompt = """Answer this question using the ReAct framework.

You have access to these tools:
- search(query): Search the web for information
- calculate(expression): Evaluate a math expression
- lookup(term): Look up a specific fact

Format EVERY step as:
Thought N: [your reasoning about what to do next]
Action N: tool_name("argument")
Observation N: [simulate a realistic result]

Continue until you can give a Final Answer.

Question: What is the total GDP of the three most populous countries in the world, and how does it compare to the entire EU's GDP?"""

show_prompt("ReAct — Multi-step Research Question", react_prompt)

r7 = call_llm([
    {"role": "system", "content": "You are a research agent. Follow the ReAct framework strictly. "
     "Simulate tool usage with realistic results. Show your complete reasoning chain."},
    {"role": "user", "content": react_prompt}
], temperature=0.3, max_tokens=1500)

show_response(r7["content"], r7)

show_tip("ReAct is the foundation of LangChain agents, AutoGPT, and OpenAI function calling. "
         "In production, replace simulated observations with REAL tool calls (APIs, databases, search engines). "
         "Pure reasoning hallucinates facts; pure tools lack planning — ReAct combines both.")

---
## 8. 🎭 Role / Persona Prompting

In [ ]:
show_banner(8, "Role / Persona Prompting", "PERSONA", "#f87171",
    "Assign the model an expert identity to steer its vocabulary, depth, "
    "tone, and domain focus. The SAME question produces wildly different "
    "answers depending on the assigned role.")

question = "What causes inflation and how should governments respond?"

personas = [
    ("🎓 Economics Professor",
     "You are a Harvard economics professor specializing in macroeconomic policy. "
     "Use precise terminology, reference Keynesian/Monetarist/MMT theories, and "
     "cite historical examples. Be analytical and balanced.",
     "#60a5fa"),
    ("📰 Journalist",
     "You are a financial journalist writing for a general audience newspaper. "
     "Explain in simple terms with relatable examples (grocery prices, rent). "
     "Short sentences. Engaging and accessible. No jargon.",
     "#4ade80"),
    ("🎭 Stand-up Comedian",
     "You are a stand-up comedian doing a bit about the economy. "
     "Explain inflation using humor, absurd analogies, self-deprecating jokes. "
     "Be funny first, informative second.",
     "#fbbf24"),
]

results_8 = []
for name, system, color in personas:
    show_prompt(f"Persona: {name}", f"System: {system}\n\nQuestion: {question}", color=color)
    r = call_llm([
        {"role": "system", "content": system},
        {"role": "user", "content": question}
    ], temperature=0.7, max_tokens=400)
    results_8.append((name, r["content"], color))
    time.sleep(0.5)

show_comparison(results_8, "Same Question — Three Different Personas")

show_tip("Stack role prompting with other techniques: "
         "'You are a senior data scientist. Think step by step...' "
         "gives you domain expertise + structured reasoning.")

---
## 9. 🧠 Meta Prompting

In [11]:
show_banner(9, "Meta Prompting", "META", "#22d3ee",
    "Ask the LLM to WRITE or OPTIMIZE prompts for itself — prompting about prompting. "
    "Leverages the model's understanding of what makes effective instructions "
    "to bootstrap high-quality prompt templates.")

meta_prompt = """I'm building an AI-powered code review tool. I need a system prompt that will make an LLM analyze code and provide:
1. Bug severity rating (Critical / Major / Minor / Style)
2. Specific line references
3. Suggested fix with code
4. Explanation of WHY it's a bug

Write me a production-ready system prompt that will produce the most consistent, high-quality code reviews. Include:
- The complete system prompt
- Output format specification (JSON)
- One baked-in few-shot example
- Edge case handling instructions
- Guardrails against false positives"""

show_prompt("Meta Prompt — Generate a Code Review Prompt", meta_prompt)

r9 = call_llm([
    {"role": "system", "content": "You are a world-class prompt engineer who has optimized "
     "prompts for Fortune 500 companies. You understand LLM behavior deeply."},
    {"role": "user", "content": meta_prompt}
], temperature=0.5, max_tokens=1500)

show_response(r9["content"], r9)

show_tip("Meta prompting is the fastest way to bootstrap production prompts. "
         "The generated prompt usually needs 1-2 iterations. "
         "Feed real failures back: 'This prompt failed on X. Improve it.'")

---
## 10. 📦 Structured Output (JSON Mode)

In [ ]:
show_banner(10, "Structured Output (JSON Mode)", "PRODUCTION", "#4ade80",
    "Force the model to return valid JSON, XML, or other structured formats. "
    "Essential for production pipelines where you need to parse output programmatically.")

json_prompt = """Analyze this customer support ticket and return ONLY a JSON object.

Ticket: "Hi, I've been trying to reset my password for 3 days now. The reset email never arrives. I've checked spam. I'm a premium subscriber paying $49/month and I'm about to cancel if this isn't fixed TODAY. My account email is john@example.com. Ticket #4521."

Return JSON with this EXACT schema:
{
  "ticket_id": string,
  "customer_email": string,
  "category": "billing" | "technical" | "account" | "feature_request" | "complaint",
  "sub_category": string,
  "urgency": integer 1-5,
  "sentiment": "angry" | "frustrated" | "neutral" | "satisfied",
  "is_premium": boolean,
  "churn_risk": "low" | "medium" | "high" | "critical",
  "key_issues": [string],
  "suggested_actions": [string],
  "suggested_response": string
}"""

show_prompt("Structured JSON Extraction", json_prompt)

# ── Call with JSON mode ────────────────────────────
try:
    r10 = call_llm([
        {"role": "system", "content": "You are a customer support AI. Return ONLY valid JSON."},
        {"role": "user", "content": json_prompt}
    ], temperature=0, response_format={"type": "json_object"})
except Exception:
    r10 = call_llm([
        {"role": "system", "content": "You are a customer support AI. Return ONLY valid JSON, no markdown."},
        {"role": "user", "content": json_prompt}
    ], temperature=0)

show_response(r10["content"], r10)

# ── Validate JSON ──────────────────────────────────
try:
    raw = r10["content"].strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    parsed = json.loads(raw)

    display(HTML(f"""
    <div style="margin:12px 0; padding:14px 18px; background:rgba(74,222,128,0.08);
         border:1px solid rgba(74,222,128,0.3); border-radius:8px;
         font-family:system-ui,sans-serif;">
      <div style="color:#4ade80; font-weight:700; margin-bottom:8px;">✅ JSON Validated Successfully!</div>
      <div style="color:#8b95a5; font-size:13px;">
        Keys found: <b>{', '.join(parsed.keys())}</b><br>
        Category: <b>{parsed.get('category','?')}</b> |
        Urgency: <b>{parsed.get('urgency','?')}/5</b> |
        Churn Risk: <b>{parsed.get('churn_risk','?')}</b> |
        Premium: <b>{parsed.get('is_premium','?')}</b>
      </div>
    </div>
    """))
except json.JSONDecodeError as e:
    show_warning(f"JSON parsing failed: {e}. In production, retry with stricter instructions.")

show_tip('Use <code>response_format={"type": "json_object"}</code> with GPT-4o for guaranteed valid JSON. '
         'Always validate + retry on parse failure in production.')

---
## 11. ⛓️ Prompt Chaining

In [ ]:
show_banner(11, "Prompt Chaining", "PRODUCTION", "#60a5fa",
    "Break complex tasks into a PIPELINE of simpler prompts where each step's "
    "output feeds into the next. More reliable, testable, and debuggable than "
    "asking one mega-prompt to do everything.")

# ── Diagram ─────────────────────────────────────────
show_diagram("""
  <div style="font-weight:700; font-size:14px; color:#fff; margin-bottom:14px;">Prompt Chaining Pipeline</div>
  <div style="display:flex; align-items:center; justify-content:center; gap:8px; flex-wrap:wrap;">
    <span style="background:rgba(96,165,250,0.15); padding:10px 16px; border-radius:8px; color:#60a5fa; font-weight:600;">📄 Article</span>
    <span style="color:#4a5568; font-size:20px;">→</span>
    <span style="background:rgba(167,139,250,0.15); padding:10px 16px; border-radius:8px; color:#a78bfa; font-weight:600;">Step 1: Extract</span>
    <span style="color:#4a5568; font-size:20px;">→</span>
    <span style="background:rgba(251,146,60,0.15); padding:10px 16px; border-radius:8px; color:#fb923c; font-weight:600;">Step 2: Analyze</span>
    <span style="color:#4a5568; font-size:20px;">→</span>
    <span style="background:rgba(74,222,128,0.15); padding:10px 16px; border-radius:8px; color:#4ade80; font-weight:600;">Step 3: Summarize</span>
  </div>
""")

article = ("Quantum computing startup QuantumLeap announced a $500M Series C round today, "
    "led by Sequoia Capital with participation from Google Ventures and SoftBank. "
    "The company claims its 1000-qubit processor can solve optimization problems "
    "100x faster than classical supercomputers. CEO Dr. Priya Sharma stated that "
    "commercial applications in drug discovery and financial modeling are expected "
    "by Q3 2026. Critics, including MIT professor Dr. James Wright, argue the "
    "claims are 'wildly optimistic' and that error correction remains unsolved. "
    "The company plans to use the funding to build a 10,000-qubit system.")

show_prompt("Input Article", article, color="#60a5fa")

# ── STEP 1: Extract ────────────────────────────────
display(HTML('<div style="margin:16px 0 6px; font-size:14px; font-weight:700; color:#a78bfa;">🔗 CHAIN STEP 1 — Entity Extraction</div>'))

r11a = call_llm([
    {"role": "system", "content": "Extract entities from text. Return JSON only."},
    {"role": "user", "content": f"Extract all named entities. Return JSON with keys: "
     f"companies, people, amounts, technologies, dates.\n\n{article}"}
], temperature=0)
show_response(r11a["content"], r11a, "Step 1 Output: Entities", "#a78bfa")

time.sleep(0.5)

# ── STEP 2: Analyze ────────────────────────────────
display(HTML('<div style="margin:16px 0 6px; font-size:14px; font-weight:700; color:#fb923c;">🔗 CHAIN STEP 2 — Sentiment Per Entity</div>'))

r11b = call_llm([
    {"role": "system", "content": "You are a sentiment analyst."},
    {"role": "user", "content": f"Analyze sentiment toward each entity (positive/negative/neutral) "
     f"with evidence.\n\nArticle: {article}\n\nEntities: {r11a['content']}\n\n"
     f"Return JSON mapping entity → sentiment + evidence."}
], temperature=0)
show_response(r11b["content"], r11b, "Step 2 Output: Sentiment", "#fb923c")

time.sleep(0.5)

# ── STEP 3: Summarize ──────────────────────────────
display(HTML('<div style="margin:16px 0 6px; font-size:14px; font-weight:700; color:#4ade80;">🔗 CHAIN STEP 3 — Executive Summary</div>'))

r11c = call_llm([
    {"role": "system", "content": "You write 3-sentence executive summaries for busy investors."},
    {"role": "user", "content": f"Using the article and analysis below, write a 3-sentence "
     f"executive summary: (1) key event, (2) bull case, (3) bear case.\n\n"
     f"Article: {article}\n\nSentiment: {r11b['content']}"}
], temperature=0.5, max_tokens=300)
show_response(r11c["content"], r11c, "Step 3 Output: Executive Summary", "#4ade80")

show_tip("Chaining advantages: each step is simpler, more testable, independently debuggable. "
         "You can cache intermediates, swap models per step (GPT-4o for reasoning, "
         "GPT-3.5 for formatting), and add validation between steps.")

---
## 📊 Session Summary & Cost Report

In [ ]:
display(HTML(f"""
<div style="margin:30px 0; padding:24px; background:#0e1117;
     border:2px solid #4ade80; border-radius:12px;
     font-family:system-ui,sans-serif;">
  <h2 style="color:#4ade80; margin:0 0 16px; font-size:18px;">📊 Session Summary</h2>
  <table style="width:100%; font-size:14px; color:#c9d1d9;">
    <tr><td style="padding:6px 0; color:#8b95a5;">Model used:</td>
        <td style="padding:6px 0; font-weight:700;">{MODEL}</td></tr>
    <tr><td style="padding:6px 0; color:#8b95a5;">Total API calls:</td>
        <td style="padding:6px 0; font-weight:700;">{usage_log['calls']}</td></tr>
    <tr><td style="padding:6px 0; color:#8b95a5;">Total tokens used:</td>
        <td style="padding:6px 0; font-weight:700;">{usage_log['tokens']:,}</td></tr>
    <tr><td style="padding:6px 0; color:#8b95a5;">Estimated total cost:</td>
        <td style="padding:6px 0; font-weight:700; color:#4ade80;">${usage_log['cost']:.4f}</td></tr>
  </table>
</div>
"""))

---
## 🧪 Playground — Try Your Own Prompts

Use the helper function below to quickly test any technique:

In [ ]:
# ═══════════════════════════════════════════════════════
#  PLAYGROUND — Edit & run your own experiments!
# ═══════════════════════════════════════════════════════

# Change these to try different techniques:
SYSTEM = "You are a helpful assistant. Think step by step."

USER = """If a train leaves Station A at 9:00 AM traveling at 60 mph,
and another train leaves Station B (300 miles away) at 10:00 AM traveling
at 90 mph toward Station A, at what time do they meet?

Let's think step by step."""

# Run it
show_prompt("Your Prompt", f"System: {SYSTEM}\n\nUser: {USER}")
result = call_llm([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": USER}
], temperature=0.3)
show_response(result["content"], result)

---
## 📚 Quick Reference Cheatsheet

In [ ]:
display(HTML("""
<div style="font-family:system-ui,sans-serif; margin:20px 0;">
<h3 style="color:#fff;">📚 Prompt Engineering Cheatsheet</h3>
<table style="width:100%; border-collapse:collapse; font-size:13px;">
  <thead>
    <tr style="background:#161b22;">
      <th style="text-align:left; padding:10px; color:#4ade80; border-bottom:2px solid #30363d;">Technique</th>
      <th style="text-align:left; padding:10px; color:#4ade80; border-bottom:2px solid #30363d;">Examples?</th>
      <th style="text-align:left; padding:10px; color:#4ade80; border-bottom:2px solid #30363d;">Best For</th>
      <th style="text-align:left; padding:10px; color:#4ade80; border-bottom:2px solid #30363d;">Cost</th>
      <th style="text-align:left; padding:10px; color:#4ade80; border-bottom:2px solid #30363d;">Key Trick</th>
    </tr>
  </thead>
  <tbody style="color:#c9d1d9;">
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Zero-Shot</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">None</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Simple classification, extraction</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Clear, specific instructions</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Few-Shot</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">2-5</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Format matching, style transfer</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Diverse, high-quality examples</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Chain-of-Thought</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">1-3 with steps</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Math, logic, multi-step</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Show worked examples</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Zero-Shot CoT</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">None</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Quick reasoning boost</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">"Let's think step by step"</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Self-Consistency</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Same × N</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">High-stakes correctness</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰💰💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Majority vote across N paths</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Tree of Thoughts</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Instructions</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Puzzles, planning, creative</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰💰💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Branch, evaluate, backtrack</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>ReAct</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Tool definitions</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Research, agents, tool use</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Thought→Action→Observation loop</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Role Prompting</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">None</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Domain expertise, tone</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">"You are a [expert role]..."</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Meta Prompting</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">None</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Prompt optimization</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">"Write me a prompt that..."</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #21262d;"><b>Structured Output</b></td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Schema</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">Production pipelines</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">💰</td>
        <td style="padding:8px; border-bottom:1px solid #21262d;">JSON mode + schema in prompt</td></tr>
    <tr><td style="padding:8px;"><b>Prompt Chaining</b></td>
        <td style="padding:8px;">Per step</td>
        <td style="padding:8px;">Complex multi-task workflows</td>
        <td style="padding:8px;">💰💰</td>
        <td style="padding:8px;">Output of step N → input of step N+1</td></tr>
  </tbody>
</table>

<div style="margin-top:16px; padding:12px 16px; background:rgba(74,222,128,0.06);
     border-left:3px solid #4ade80; border-radius:0 8px 8px 0; font-size:13px; color:#b0d4b8;">
  💡 <b style="color:#4ade80;">Golden Rule:</b> Start simple (Zero-Shot) → add examples if needed (Few-Shot) →
  add reasoning for complex tasks (CoT) → scale to ToT/Self-Consistency only when accuracy demands it.
  Don't over-engineer your prompts!
</div>
</div>
"""))